# 07 - Quantization benchmark + API demo (Week 4)

**Part A:** load the same fine-tuned VLM three ways (4-bit, 8-bit, 16-bit) and compare GPU memory, speed and accuracy on the same 150 eval images (10 per category: 7 defective + 3 good).
**Part B:** run the FastAPI service (`src/defect_vlm/api.py`) against the real 4-bit model and send it images.

**Settings:** GPU T4 x2 · Internet On · Inputs: **MVTec AD**, **mvtec-split**, **mvtec-finetune**.
**Time:** roughly 30-50 minutes → **Save Version → Save & Run All**, then close the tab.

Scope note: the original plan said AWQ-INT4 served with vLLM. This uses bitsandbytes 4-bit/8-bit through the same Unsloth path as training, which is far less likely to break on a T4. vLLM/AWQ is listed as future work.

In [ ]:
%%capture
!pip install -q unsloth fastapi python-multipart httpx

In [ ]:
import os, sys, gc, time, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"
print(torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
from pathlib import Path

split_files = list(Path('/kaggle/input').rglob('splits.csv'))
assert split_files, "splits.csv not found: Add Input -> Your Work -> mvtec-split"
df = pd.read_csv(split_files[0])
roots = [p.parent for p in Path('/kaggle/input').rglob('bottle') if p.is_dir() and (p / 'train').exists()]
assert roots, "MVTec AD not found: Add Input -> the MVTec AD dataset"
ROOT = roots[0]

def rebase(p):
    if not isinstance(p, str):
        return p
    parts = p.split('/')
    i = parts.index('ground_truth') - 1 if 'ground_truth' in parts else len(parts) - 4
    return str(ROOT.joinpath(*parts[i:]))

df['path'] = df['path'].map(rebase)
df['mask_path'] = df['mask_path'].map(rebase)
assert Path(df.path.iloc[0]).exists(), "paths don't resolve"
print(len(df), 'rows |', df.role.value_counts().to_dict())

In [ ]:
cfgs = list(Path('/kaggle/input').rglob('adapter_config.json'))
assert cfgs, "Adapters not found: Add Input -> Your Work -> mvtec-finetune"
ADAPTER_DIR = '/kaggle/working/qwen2vl-mvtec-lora'
shutil.copytree(cfgs[0].parent, ADAPTER_DIR, dirs_exist_ok=True)
os.makedirs('/kaggle/working/defect_vlm', exist_ok=True)
print('adapters ready')

## The project code (identical to `src/defect_vlm/` in the repo)

In [ ]:
%%writefile /kaggle/working/defect_vlm/inference.py
"""Run the fine-tuned Qwen2-VL-2B inspector on one image.

Heavy libraries (torch, unsloth) are imported only when an Inspector is created,
so the API and its tests can be imported on a laptop without a GPU.
"""
from __future__ import annotations

import json
import math

from PIL import Image

IMG_SIZE = 448  # must match training

DEFECT_TYPES = {
    "bottle": ["broken_large", "broken_small", "contamination"],
    "cable": ["bent_wire", "cable_swap", "combined", "cut_inner_insulation", "cut_outer_insulation",
              "missing_cable", "missing_wire", "poke_insulation"],
    "capsule": ["crack", "faulty_imprint", "poke", "scratch", "squeeze"],
    "carpet": ["color", "cut", "hole", "metal_contamination", "thread"],
    "grid": ["bent", "broken", "glue", "metal_contamination", "thread"],
    "hazelnut": ["crack", "cut", "hole", "print"],
    "leather": ["color", "cut", "fold", "glue", "poke"],
    "metal_nut": ["bent", "color", "flip", "scratch"],
    "pill": ["color", "combined", "contamination", "crack", "faulty_imprint", "pill_type", "scratch"],
    "screw": ["manipulated_front", "scratch_head", "scratch_neck", "thread_side", "thread_top"],
    "tile": ["crack", "glue_strip", "gray_stroke", "oil", "rough"],
    "toothbrush": ["defective"],
    "transistor": ["bent_lead", "cut_lead", "damaged_case", "misplaced"],
    "wood": ["color", "combined", "hole", "liquid", "scratch"],
    "zipper": ["broken_teeth", "combined", "fabric_border", "fabric_interior", "rough",
               "split_teeth", "squeezed_teeth"],
}
LOCATIONS = ["top-left", "top", "top-right", "left", "center", "right",
             "bottom-left", "bottom", "bottom-right", "none"]


def make_prompt(category: str) -> str:
    name = category.replace("_", " ")
    types = ", ".join(sorted(DEFECT_TYPES[category]))
    return (f"You are inspecting a {name} on a factory line. Is it defective? "
            f"Possible defect types: {types}. "
            'Answer only with JSON: {"defective": true/false, "defect_type": "<type or none>", '
            '"location": "<top-left|top|top-right|left|center|right|bottom-left|bottom|bottom-right|none>"}')


def prepare_image(image: Image.Image) -> Image.Image:
    return image.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)


def parse_json(text: str):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        return json.loads(text[start:end])
    except ValueError:
        return None


class Inspector:
    """Loads base model + LoRA adapters once; call .inspect(image, category) per request."""

    MODES = ("4bit", "8bit", "16bit")

    def __init__(self, adapter_dir: str, mode: str = "4bit"):
        if mode not in self.MODES:
            raise ValueError(f"mode must be one of {self.MODES}")
        import torch
        from unsloth import FastVisionModel

        self.torch = torch
        kwargs = {"4bit": dict(load_in_4bit=True),
                  "8bit": dict(load_in_4bit=False, load_in_8bit=True),
                  "16bit": dict(load_in_4bit=False, dtype=torch.float16)}[mode]
        self.mode = mode
        self.model, self.processor = FastVisionModel.from_pretrained(adapter_dir, **kwargs)
        FastVisionModel.for_inference(self.model)
        self.tok = getattr(self.processor, "tokenizer", self.processor)
        self.true_id = self.tok.encode(" true", add_special_tokens=False)[0]
        self.false_id = self.tok.encode(" false", add_special_tokens=False)[0]

    def _inputs(self, image, category, prefix=""):
        msgs = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": make_prompt(category)}]}]
        text = self.processor.apply_chat_template(msgs, add_generation_prompt=True) + prefix
        return self.processor(prepare_image(image), text, add_special_tokens=False,
                              return_tensors="pt").to("cuda")

    def _prob_true(self, logits):
        pair = logits[[self.true_id, self.false_id]].float()
        return self.torch.softmax(pair, 0)[0].item()

    def inspect(self, image: Image.Image, category: str) -> dict:
        if category not in DEFECT_TYPES:
            raise ValueError(f"unknown category: {category}")
        torch = self.torch
        with torch.no_grad():
            inputs = self._inputs(image, category)
            out = self.model.generate(**inputs, max_new_tokens=48, do_sample=False, use_cache=True,
                                      return_dict_in_generate=True, output_scores=True)
            gen = out.sequences[0, inputs["input_ids"].shape[1]:]
            score = float("nan")
            if out.scores is not None:
                for step, tid in enumerate(gen.tolist()):
                    if tid in (self.true_id, self.false_id):
                        score = self._prob_true(out.scores[step][0])
                        break
            if math.isnan(score):  # backup: read the probability directly
                inputs = self._inputs(image, category, prefix='{"defective":')
                score = self._prob_true(self.model(**inputs).logits[0, -1])
        raw = self.tok.decode(gen, skip_special_tokens=True).strip()
        pred = parse_json(raw) or {}
        return {"defective": pred.get("defective"), "defect_type": pred.get("defect_type"),
                "location": pred.get("location"), "score": round(score, 4),
                "valid_json": bool(pred), "raw": raw}


In [ ]:
%%writefile /kaggle/working/defect_vlm/api.py
"""FastAPI service: POST a photo + its category, get the inspection as JSON.

Run on a GPU machine:
    ADAPTER_DIR=/path/to/qwen2vl-mvtec-lora QUANT=4bit uvicorn defect_vlm.api:app
"""
from __future__ import annotations

import io
import os
import time
from typing import Optional

from fastapi import Depends, FastAPI, File, Form, HTTPException, UploadFile
from PIL import Image, UnidentifiedImageError
from pydantic import BaseModel

from defect_vlm.inference import DEFECT_TYPES

app = FastAPI(title="VLM defect inspector",
              description="Fine-tuned Qwen2-VL-2B on MVTec AD. Returns defective/type/location plus a score.")

_inspector = None


def get_inspector():
    """Load the model once, on first request. Tests replace this with a fake."""
    global _inspector
    if _inspector is None:
        from defect_vlm.inference import Inspector
        _inspector = Inspector(os.environ["ADAPTER_DIR"], mode=os.environ.get("QUANT", "4bit"))
    return _inspector


class Inspection(BaseModel):
    category: str
    defective: Optional[bool]
    defect_type: Optional[str]
    location: Optional[str]
    score: float
    valid_json: bool
    latency_ms: float


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/categories")
def categories():
    return DEFECT_TYPES


@app.post("/inspect", response_model=Inspection)
async def inspect(category: str = Form(...), image: UploadFile = File(...),
                  inspector=Depends(get_inspector)):
    if category not in DEFECT_TYPES:
        raise HTTPException(400, f"Unknown category '{category}'. See GET /categories.")
    try:
        img = Image.open(io.BytesIO(await image.read()))
        img.load()
    except (UnidentifiedImageError, OSError):
        raise HTTPException(400, "The uploaded file is not a readable image.")
    t0 = time.perf_counter()
    result = inspector.inspect(img, category)
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)
    return Inspection(category=category, latency_ms=latency_ms,
                      **{k: result[k] for k in ("defective", "defect_type", "location", "score", "valid_json")})


In [ ]:
open('/kaggle/working/defect_vlm/__init__.py', 'w').close()
sys.path.insert(0, '/kaggle/working')
from defect_vlm.inference import Inspector, DEFECT_TYPES
from PIL import Image

actual = df[df.label == 'defect'].groupby('category').defect_type.unique().map(sorted).to_dict()
assert actual == {k: sorted(v) for k, v in DEFECT_TYPES.items()}, 'DEFECT_TYPES does not match the dataset'
print('defect taxonomy matches the dataset ✓')

## Part A - Quantization benchmark
Same 150 eval images for every variant. Accuracy here is on a small subset, so compare the variants with each other, not with the Week 3 scorecard.

In [ ]:
ev = df[df.role == 'eval']
subset = pd.concat([g.sample(min(7 if lab == 'defect' else 3, len(g)), random_state=0)
                    for (cat, lab), g in ev.groupby(['category', 'label'])]).reset_index(drop=True)
MODES = ['4bit', '8bit', '16bit']
print(len(subset), 'images |', subset.label.value_counts().to_dict())

In [ ]:
import transformers, numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
transformers.logging.set_verbosity_error()

def macro_auroc(d):
    vals = [roc_auc_score(g.label == 'defect', g.score) for _, g in d.groupby('category') if g.label.nunique() == 2]
    return float(np.mean(vals))

def summarise(p):
    y = p.label == 'defect'
    said_defect = p.pred_defective.astype(str).str.lower().eq('true')
    return dict(macro_auroc=round(macro_auroc(p), 3),
                defective_acc=round(float((said_defect == y).mean()), 3),
                type_acc=round(float((p[y].pred_type == p[y].true_type).mean()), 3),
                valid_json=round(float(p.valid_json.mean()), 3))

results, all_preds = [], []
for mode in MODES:
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    try:
        insp = Inspector(ADAPTER_DIR, mode=mode)
    except Exception as e:
        print(f'{mode}: could not load -> {type(e).__name__}: {e}')
        results.append({'mode': mode, 'status': f'load failed: {type(e).__name__}'})
        continue
    row = {'mode': mode, 'status': 'ok', 'load_gb': round(torch.cuda.memory_allocated() / 1e9, 2)}
    for _, r in subset.head(2).iterrows():                      # warm-up, not timed
        insp.inspect(Image.open(r.path), r.category)
    lat, preds = [], []
    for _, r in tqdm(subset.iterrows(), total=len(subset), desc=mode):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        out = insp.inspect(Image.open(r.path), r.category)
        torch.cuda.synchronize(); lat.append((time.perf_counter() - t0) * 1000)
        preds.append({'mode': mode, 'path': r.path, 'category': r.category, 'label': r.label,
                      'true_type': r.defect_type, 'pred_defective': out['defective'],
                      'pred_type': out['defect_type'], 'pred_location': out['location'],
                      'score': out['score'], 'valid_json': out['valid_json']})
    p = pd.DataFrame(preds); all_preds += preds
    row.update(peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
               median_ms=round(float(np.median(lat))), p90_ms=round(float(np.percentile(lat, 90))),
               **summarise(p))
    results.append(row); print(row)
    del insp; gc.collect(); torch.cuda.empty_cache()

quant = pd.DataFrame(results)
quant.to_csv('/kaggle/working/quant_results.csv', index=False)
pd.DataFrame(all_preds).to_csv('/kaggle/working/quant_preds.csv', index=False)
quant

## Part B - The API, called with real images
The same FastAPI app that's in the repo, served in-process against the real 4-bit model.

In [ ]:
from fastapi.testclient import TestClient
import io, json
from defect_vlm import api

insp = Inspector(ADAPTER_DIR, mode='4bit')
api.app.dependency_overrides[api.get_inspector] = lambda: insp
client = TestClient(api.app)

print('GET /health ->', client.get('/health').json())
demo = []
for _, r in subset.sample(5, random_state=3).iterrows():
    with open(r.path, 'rb') as f:
        resp = client.post('/inspect', data={'category': r.category},
                           files={'image': (Path(r.path).name, f, 'image/png')})
    body = resp.json()
    demo.append({'truth': r.defect_type, **body})
    print(f"[{r.category} | truth: {r.defect_type}] {resp.status_code} -> {json.dumps(body)}")

bad = client.post('/inspect', data={'category': 'banana'}, files={'image': ('x.png', b'not an image', 'image/png')})
print('bad request ->', bad.status_code, bad.json())
json.dump(demo, open('/kaggle/working/api_demo.json', 'w'), indent=2)